<a href="https://colab.research.google.com/github/pop123-ux/Qwen2.5-7B-Instruct_finetuned_for_seedance2.0_prompting/blob/main/cod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install trl
!pip3 install deepspeed
!pip install --upgrade torchao

In [4]:
import pandas as pd
import requests
import io

# Updated URL to directly resolve the raw file content, including Git LFS files
url = "https://huggingface.co/datasets/GokuScraper/seedance-2-prompts-datasets/resolve/main/metadata.jsonl"

response = requests.get(url)
response.raise_for_status() # Raise an exception for HTTP errors
content = response.text # Get the content as a string

df = pd.read_json(io.StringIO(content), lines=True)

print(f"✅ Loaded {len(df)} structured video prompts!")

✅ Loaded 8484 structured video prompts!


In [ ]:
df["duration"] = df["spec"].apply(lambda x: x.get("duration"))
df["ratio"] = df["spec"].apply(lambda x: x.get("ratio"))
df["width"] = df["spec"].apply(lambda x: x.get("width"))
df["height"] = df["spec"].apply(lambda x: x.get("height"))

In [ ]:
df['duration'].head()

,duration
0,15.10
1,15.13
2,15.12
3,15.10
4,15.13


In [5]:
from datasets import Dataset

ds = Dataset.from_pandas(df)
ds

Dataset({
    features: ['category', 'date', 'file_name', 'i18n', 'id', 'is_featured', 'media', 'model_info', 'platform', 'raw_p', 'slug', 'sourceLink', 'spec', 'version'],
    num_rows: 8484
})

In [ ]:
ds.features

{'category': Value('string'),
 'date': Value('timestamp[ns]'),
 'file_name': Value('string'),
 'i18n': {'en': {'p': Value('string'),
   't': Value('string'),
   'tags': List(Value('string'))},
  'zh': {'p': Value('string'),
   't': Value('string'),
   'tags': List(Value('string'))}},
 'id': Value('string'),
 'is_featured': Value('bool'),
 'media': {'c': Value('string'),
  'ref_images': List(Value('string')),
  'v': Value('string')},
 'model_info': {'model': Value('string'),
  'name': Value('string'),
  'version': Value('string')},
 'platform': Value('string'),
 'raw_p': Value('string'),
 'slug': Value('string'),
 'sourceLink': Value('string'),
 'spec': {'duration': Value('float64'),
  'height': Value('int64'),
  'ratio': Value('float64'),
  'safety_rating': Value('string'),
  'width': Value('int64')},
 'version': Value('int64'),
 'duration': Value('float64'),
 'ratio': Value('float64'),
 'width': Value('int64'),
 'height': Value('int64')}

In [ ]:
ds['spec']['height']

Column([720, 720, 720, 720, 720])

In [6]:
# Map the dataset to our needs

def convert(example):
    en = example["i18n"].get("en", {})
    spec = example["spec"]

    prompt = en.get("p") or example.get("raw_p") or ""

    return {
        "messages": [
            {
                "role": "user",
                "content": (
                    f"Write a Seedance 2 cinematic prompt.\n"
                    f"Category: {example['category']}\n"
                    f"Duration: {spec['duration']}\n"
                    f"Aspect Ratio: {spec['ratio']}"
                ),
            },
            {
                "role": "assistant",
                "content": prompt,
            },
        ]
    }

dataset = ds.map(convert)

Map:   0%|          | 0/8484 [00:00<?, ? examples/s]

In [7]:
# The part of the dataset that contains empty columns cannot be properly parsed by the "tokenize" function we're about to write

for i, ex in enumerate(dataset):
    for msg in ex["messages"]:
        if msg["content"] is None:
            print(i)
            print(ex)
            break

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    'Qwen/Qwen2.5-1.5B-Instruct'
)

# The 'tokenize' function I was writing about before
def tokenize(example):
  text = tokenizer.apply_chat_template(
      example['messages'],
      tokenize=False,
      add_generation_prompt=False
  )

  return tokenizer(
      text,
      truncation=True,
      max_length=4096
  )

dataset_f = dataset.map(tokenize)

Map:   0%|          | 0/8484 [00:00<?, ? examples/s]

In [ ]:
!accelerate config

--------------------------------------------------------------------------------In which compute environment are you running?
Please input a choice index (starting from 0), and press enter
 ➔  This machine
    AWS (Amazon SageMaker)

This machine
--------------------------------------------------------------------------------Which type of machine are you using?
Please input a choice index (starting from 0), and press enter
 ➔  No distributed training
    multi-CPU
    multi-XPU
    multi-HPU
    multi-GPU
    multi-NPU
    multi-MLU
    multi-SDAA
    multi-MUSA
    multi-NEURON
    TPU

No distributed training
Do you want to run your training on CPU only (even if a GPU / Apple Silicon / Ascend NPU device is available)? [yes/NO]:NO
Do you wish to optimize your script with torch dynamo?[yes/NO]:yes
--------------------------------------------------------------------------------Which dynamo backend would you like to use?
Please input a choice index (starting from 0), and press enter
    

In [9]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map='auto',
    dtype='bfloat16'
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [10]:
# Add LoRA
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_f,
    args=SFTConfig(
        output_dir="./seedance-with-lora",
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        bf16=True,
        logging_steps=10,
        save_steps=500,
        packing=False,
        max_seq_length=1024,
    ),
)

model.config.use_cache = False

trainer.train()

Building labels for train dataset:   0%|          | 0/8484 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8484 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8484 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.895469
20,2.438393
30,2.410674
40,2.442296
50,2.388594
60,2.406348
70,2.344794
80,2.356938
90,2.305909
100,2.397007


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub('pop123-ux/seedance-qwen2.5-7b-lora')
tokenizer.push_to_hub('pop123-ux/seedance-qwen2.5-7b-lora')

In [ ]:
# Load model example
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model = PeftModel.from_pretrained(
    base,
    "your-username/seedance-qwen2.5-7b-lora"
)

tokenizer = AutoTokenizer.from_pretrained("your-username/seedance-qwen2.5-7b-lora")